> **中文阅读版**：本文件是 `08_pipeline_verification.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# 流水线验证：回测与实盘一致性

**Docker 镜像**：`ml4t`

**书籍参考**：第 25 章，第 25.6 节（通过流水线验证确保技术一致性）

[`01_unified_framework_demo`](01_unified_framework_demo.ipynb) 针对交叉策略所产生的一项内容——信号——比较了两个引擎。这足以展示想法，但不足以部署。实盘流水线可以就信号达成一致，却仍可能在产生信号的特征、特征所输入的预测，或信号转化成的订单规模上不一致，而其中每一种失败都会触及本书技术栈的不同部分。

因此，一致性需要逐阶段检查，而每个阶段都是一道门禁，而不是一份报告。来自相同 bar 的特征必须完全一致。来自相同特征的预测必须完全一致。来自相同信号的订单规模必须完全一致。仅仅打印其不一致之处的测试，是没人会注意到其失败的测试，这也是下面这套测试会统计其门禁数量、并由最后一个单元格对该数量进行断言的原因。

**学习目标**
- 将一致性声明拆分为多个阶段，使失败能够指明是哪一层出了问题
- 构建确定性的行情带（tape），让两条流水线都能在其上运行，且不依赖读者机器所控制的任何东西
- 将不应存在的差异与预期且必须声明的差异区分开来
- 留下一套会让持续集成运行失败的测试套件，而不是一套仅仅描述自身的套件

**前置条件**：[`01_unified_framework_demo`](01_unified_framework_demo.ipynb) 提供了该比较的单阶段版本。

In [1]:
"""Pipeline Verification: stage-by-stage backtest vs live parity checks."""

import asyncio
import logging
import os
import sys
import tempfile
import warnings
from collections import deque
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any

# The broker adapters pull in websockets' legacy module, which deprecates itself on import.
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"websockets\.legacy")

import numpy as np
from async_utils import run_async
from ml4t.backtest import OrderSide, Strategy
from ml4t.backtest.types import Order, OrderStatus, OrderType
from ml4t.live import LiveRiskConfig
from ml4t.live.safety import SafeBroker, VirtualPortfolio
from ml4t.live.wrappers import ThreadSafeBrokerWrapper

from utils.reproducibility import set_global_seeds

# force=True is deliberate: an imported library may already have attached a root handler, and
# without it basicConfig would silently do nothing and the notebook's log lines would not appear.
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s - %(message)s",
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger("pipeline_verification")

print("[OK] Components imported")

[OK] Components imported


## 设置

`N_BARS` 表示合成行情带（tape）包含多少根 bar。30 足以让最长的特征窗口完成预热，并且仍为每个阶段留下可供比较的 bar，同时又足够小，使失败可以逐行读取。

`SEED` 固定该行情带。一致性测试比较的是两条流水线彼此之间，因此重要的不是它们拿到哪一条行情带，而是它们每次、在每台机器上都拿到同一条行情带。

In [2]:
N_BARS = 30
SEED = 12345

In [3]:
set_global_seeds(SEED)

**学习目标**
- 构建确定性数据，使其能够同时驱动回测与实盘流水线。
- 逐阶段比较特征、预测和订单逻辑。
- 将一致性检查转化为回归测试，在行为发生偏离时明确失败。

**发现：** 将相同的回测组件和实盘组件导入同一个 notebook，就是第一项一致性测试。如果这些基础设施无法共存于单一环境，那么在本章的统一框架主张在任何策略逻辑运行之前就已经不成立了。

## 1. 定义测试策略

该策略被有意设计为确定性的，因此回测输出与实盘输出之间的任何不一致都是技术缺陷，而不是统计波动。

In [4]:
@dataclass
class VerificationResult:
    """Result of a single verification test."""

    test_name: str
    passed: bool
    expected: Any
    actual: Any
    message: str = ""
    skipped: bool = False
    expected_difference: bool = False


# A parity harness must not waive a mismatch merely because it is convenient.
# Any intentional divergence belongs in a separate, explicitly tested contract.
EXPECTED_DIFFERENCE: set[str] = set()

### 确定性策略函数

这些纯函数隔离出两条执行路径都必须复现的四项决策。

In [5]:
def compute_features(prices: list[float], lookback: int) -> dict[str, float]:
    """Compute the feature vector from the latest complete lookback window."""
    if len(prices) < lookback:
        return {}
    values = np.asarray(prices[-lookback:])
    returns = np.diff(values) / values[:-1]
    return {
        "momentum": (values[-1] / values[0]) - 1,
        "volatility": float(np.std(returns)),
        "mean": float(np.mean(values)),
        "current": float(values[-1]),
    }

该预测是一条固定线性规则，因此不一致只能来自实现漂移。

In [6]:
def compute_prediction(features: dict[str, float]) -> float:
    """Map a complete feature vector to one deterministic score."""
    if not features:
        return 0.0
    return (
        features["momentum"] * 0.5
        - features["volatility"] * 0.3
        + (features["current"] / features["mean"] - 1) * 0.2
    )

持仓状态对入场和出场进行门控；HOLD 是不交易区间内唯一的动作。

In [7]:
def compute_signal(prediction: float, threshold: float, has_position: bool) -> str:
    """Convert a score and current position state to an order intent."""
    if prediction > threshold and not has_position:
        return "BUY"
    if prediction < -threshold and has_position:
        return "SELL"
    return "HOLD"

固定比例仓位规模使一致性演练聚焦于相同的输入和状态。

In [8]:
def compute_size(signal: str, price: float, cash: float, position_quantity: int) -> int:
    """Size entries from cash and bound exits by the current long position."""
    if signal == "HOLD":
        return 0
    target = max(0, int(cash * 0.1 / price))
    return target if signal == "BUY" else min(target, max(0, position_quantity))

成交记录捕获提交后已实现的订单和券商状态。

In [9]:
def fill_record(
    timestamp: datetime,
    symbol: str,
    signal: str,
    size: int,
    price: float,
    order: Order,
    broker: Any,
) -> dict[str, Any]:
    positions = tuple(
        sorted((asset, int(item.quantity)) for asset, item in broker.positions.items())
    )
    return {
        "timestamp": timestamp,
        "symbol": symbol,
        "signal": signal,
        "size": size,
        "price": price,
        "order_id": order.order_id,
        "status": order.status.value,
        "filled_quantity": int(order.filled_quantity),
        "filled_price": float(order.filled_price),
        "cash": float(broker.get_cash()),
        "positions": positions,
    }

按 symbol 的步骤在提交可执行意图之前记录每一个中间值。

In [10]:
def process_symbol(strategy: Any, timestamp: datetime, symbol: str, bar: dict, broker: Any) -> None:
    """Advance one symbol through the complete decision pipeline."""
    prices = strategy.prices.setdefault(symbol, deque(maxlen=strategy.lookback + 5))
    close = bar["close"]
    prices.append(close)
    features = compute_features(list(prices), strategy.lookback)
    strategy.feature_log.append(
        {"timestamp": timestamp, "symbol": symbol, "features": features.copy()}
    )
    if not features:
        return

    prediction = compute_prediction(features)
    strategy.prediction_log.append(
        {"timestamp": timestamp, "symbol": symbol, "prediction": prediction}
    )
    position = broker.get_position(symbol)
    signal = compute_signal(
        prediction, strategy.threshold, bool(position and position.quantity > 0)
    )
    strategy.signal_log.append(
        {"timestamp": timestamp, "symbol": symbol, "signal": signal, "prediction": prediction}
    )
    if signal == "HOLD":
        return

    cash = broker.get_cash()
    position_quantity = int(position.quantity) if position else 0
    size = compute_size(signal, close, cash, position_quantity)
    if size <= 0:
        return
    side = OrderSide.BUY if signal == "BUY" else OrderSide.SELL
    order = broker.submit_order(symbol, size, side=side)
    strategy.order_log.append(fill_record(timestamp, symbol, signal, size, close, order, broker))

### 可验证策略

该策略持有状态，并将每个 symbol 委托给上面的可观察决策步骤。

In [11]:
class VerifiableStrategy(Strategy):
    """Strategy with stage logs for backtest-live comparison."""

    def __init__(self, lookback: int = 10, threshold: float = 0.02):
        self.lookback = lookback
        self.threshold = threshold
        self.prices: dict[str, deque] = {}
        self.feature_log: list[dict] = []
        self.prediction_log: list[dict] = []
        self.signal_log: list[dict] = []
        self.order_log: list[dict] = []

    def on_start(self, broker: Any) -> None:
        logger.info("VerifiableStrategy started")

    def on_data(self, timestamp: datetime, data: dict, context: dict, broker: Any) -> None:
        for symbol, bar in data.items():
            process_symbol(self, timestamp, symbol, bar, broker)

    def on_end(self, broker: Any) -> None:
        logger.info("Strategy ended: %s orders", len(self.order_log))

## 2. 创建测试数据

回测与实盘都会处理的确定性测试数据。

确定性在这里很重要，因为任何随机性都会削弱不一致与其产生所经代码路径之间的因果联系。该 notebook 试图隔离的是技术性差异，而不是市场噪声。

一致性测试的好坏取决于两条流水线所读取的行情带，因此该行情带在每台运行它的机器上都必须完全相同。为生成器设置种子只是容易做到的那一半。更难的那一半是：从 Python 内置 `hash()` 推导出的按 symbol 偏移量并不可靠：字符串上的 `hash()` 会按进程随机化，除非解释器以固定的 `PYTHONHASHSEED` 启动，而 notebook 无法自行设置它。这样一来，两次运行会读取两条不同的行情带，任何不一致都将无法归因。因此，偏移量改为显式写出。

In [12]:
set_global_seeds(SEED)

SYMBOLS = ["SPY", "QQQ"]
SYMBOL_OFFSETS = {"SPY": 101, "QQQ": 202}

In [13]:
def generate_test_data() -> list[tuple[datetime, dict]]:
    """Generate deterministic OHLCV bars."""
    data = []
    base_prices = {"SPY": 500.0, "QQQ": 400.0}

    for i in range(N_BARS):
        timestamp = datetime(2025, 1, 1, 10, 0) + timedelta(minutes=i)
        bar_data = {}

        for symbol in SYMBOLS:
            np.random.seed(SEED + i * 100 + SYMBOL_OFFSETS[symbol])
            # A rising-then-falling SPY tape and its QQQ mirror guarantee that
            # the harness exercises BUY and SELL order paths, not only HOLD.
            regime = 1.0 if i < N_BARS // 2 else -1.0
            direction = regime if symbol == "SPY" else -regime
            change = direction * 0.004 + np.random.normal(0, 0.0005)
            base_prices[symbol] *= 1 + change
            price = base_prices[symbol]

            bar_data[symbol] = {
                "open": price * 0.999,
                "high": price * 1.002,
                "low": price * 0.998,
                "close": price,
                "volume": 1000000,
            }

        data.append((timestamp, bar_data))

    return data

In [14]:
TEST_DATA = generate_test_data()
print(f"[OK] Generated {len(TEST_DATA)} test bars for {SYMBOLS}")

[OK] Generated 30 test bars for ['SPY', 'QQQ']


**发现：** 上面生成的 bar 为两个引擎创建了一条共享的输入行情带。正是这条共同的行情带，使 notebook 能够将不一致归咎于实现细节，而不是不同的市场状态。

## 3. 运行回测流水线

回测运行建立了参考输出。之后的每一次比较都在问：实盘风格的路径是否能在同一合成市场行情带上复现相同的特征值、预测和订单意图。

In [15]:
class TestDataFeed:
    """Simple feed that replays test data."""

    def __init__(self, data: list):
        self._data = data
        self._index = 0
        self._running = False

    async def start(self):
        self._running = True
        self._index = 0

    def stop(self):
        self._running = False

    def __aiter__(self):
        return self

    async def __anext__(self):
        if not self._running or self._index >= len(self._data):
            raise StopAsyncIteration
        timestamp, bar_data = self._data[self._index]
        self._index += 1
        await asyncio.sleep(0.001)  # Minimal delay
        return timestamp, bar_data, {}

    @property
    def stats(self):
        return {"bars": self._index}

### 内存参考经纪商

参考经纪商按最新合成价格立即成交，并更新一个虚拟投资组合。

In [16]:
class BacktestBroker:
    """Minimal synchronous broker for the reference replay."""

    order_prefix = "BT"

    def __init__(self):
        self._portfolio = VirtualPortfolio(initial_cash=100_000.0)
        self._prices: dict[str, float] = {}
        self._next_order_id = 1

    @property
    def positions(self):
        return self._portfolio.positions

    def get_position(self, asset):
        return self._portfolio.positions.get(asset)

    def get_cash(self):
        return self._portfolio.cash

    def update_price(self, symbol, price):
        self._prices[symbol] = price

    def submit_order(self, asset, quantity, side=None, **kwargs):
        price = self._prices.get(asset, 100.0)
        order_id = f"{self.order_prefix}-{self._next_order_id:04d}"
        self._next_order_id += 1
        order = Order(
            asset=asset,
            side=side or OrderSide.BUY,
            quantity=quantity,
            order_type=OrderType.MARKET,
            order_id=order_id,
            status=OrderStatus.FILLED,
            created_at=datetime.now(),
            filled_price=price,
            filled_quantity=quantity,
        )
        self._portfolio.process_fill(order)
        return order

### 回测驱动

参考驱动记录每个阶段，不使用线程或风险控制包装器。

In [17]:
backtest_strategy = VerifiableStrategy(lookback=10, threshold=0.01)


async def run_backtest():
    """Replay the fixed tape through the reference strategy and broker."""

    broker = BacktestBroker()
    backtest_strategy.on_start(broker)

    for timestamp, bar_data in TEST_DATA:
        for symbol, bar in bar_data.items():
            broker.update_price(symbol, bar["close"])
        backtest_strategy.on_data(timestamp, bar_data, {}, broker)

    backtest_strategy.on_end(broker)

In [18]:
print("BACKTEST PIPELINE")
backtest_executed = False
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        run_async(run_backtest())
    backtest_executed = True
except RuntimeError as _e:
    if "Timeout" in str(_e) or "task" in str(_e).lower():
        print(f"Async backtest skipped (Papermill environment): {_e}")
    else:
        raise

print("\nBacktest Results:")
print(f"   Features computed: {len(backtest_strategy.feature_log)}")
print(f"   Predictions made: {len(backtest_strategy.prediction_log)}")
print(f"   Signals generated: {len(backtest_strategy.signal_log)}")
print(f"   Orders submitted: {len(backtest_strategy.order_log)}")
if backtest_strategy.order_log:
    print(f"   Final cash: ${backtest_strategy.order_log[-1]['cash']:,.2f}")
    print(f"   Final positions: {backtest_strategy.order_log[-1]['positions']}")

BACKTEST PIPELINE
INFO - VerifiableStrategy started


INFO - Shadow: Opened SPY LONG 19


INFO - Shadow: Reduced SPY to 2


INFO - Shadow: Closed SPY


INFO - Shadow: Opened QQQ LONG 25


INFO - Strategy ended: 4 orders



Backtest Results:
   Features computed: 60
   Predictions made: 42
   Signals generated: 42
   Orders submitted: 4
   Final cash: $90,250.46
   Final positions: (('QQQ', 25),)


**发现：** 回测运行在固定的合成行情序列上，为流水线的每个阶段建立参考计数。

**交易含义：** 如果没有这样的基线运行，后续实盘风格的差异就很难归类，因为不存在可用于比较的、约定好的正确输出。


## 4. 运行实时流水线（模拟）

实时流水线使用模拟基础设施，使 notebook 能够隔离框架行为，而不会引入经纪商或网络噪声。

In [19]:
print("\n" + "=" * 60)
print("LIVE PIPELINE (Simulated)")
print("=" * 60)

live_strategy = VerifiableStrategy(lookback=10, threshold=0.01)


def _temporary_state_path() -> str:
    """Return a non-existent temporary path for SafeBroker state."""
    fd, name = tempfile.mkstemp(prefix="nb08_safe_broker_", suffix=".json")
    os.close(fd)
    Path(name).unlink(missing_ok=True)
    return name


LIVE PIPELINE (Simulated)


实盘风格经纪商增加了 `SafeBroker` 所需的异步接口，同时复用与参考经纪商相同的成交记账。

In [20]:
class LiveBroker(BacktestBroker):
    """Asynchronous adapter around the in-memory reference broker."""

    order_prefix = "LIVE"

    def __init__(self):
        super().__init__()
        self._connected = False

    @property
    def execution_capabilities(self):
        return frozenset()

    async def connect(self):
        self._connected = True

    async def disconnect(self):
        self._connected = False

    async def is_connected_async(self):
        return self._connected

    @property
    def pending_orders(self):
        return []

    async def get_positions_async(self):
        return self.positions

    async def get_account_value_async(self):
        return self._portfolio.cash

    async def get_cash_async(self):
        return self._portfolio.cash

    async def submit_order_async(self, asset, quantity, side=None, **kwargs):
        return self.submit_order(asset, quantity, side=side, **kwargs)

    async def cancel_order_async(self, order_id):
        return False

    async def close_position_async(self, asset):
        return None

实时驱动增加了风险包装器和同步策略适配器，然后重放同一行情序列。

In [21]:
async def run_live():
    """Run the strategy through the live-style wrapper on the fixed tape."""
    broker = LiveBroker()
    state_path = Path(_temporary_state_path())
    risk_config = LiveRiskConfig(
        shadow_mode=True,
        max_position_value=100_000.0,
        max_order_value=50_000.0,
        dedup_window_seconds=0.0,
        state_file=str(state_path),
    )
    safe_broker = SafeBroker(broker, risk_config)
    loop = asyncio.get_running_loop()
    wrapped_broker = ThreadSafeBrokerWrapper(safe_broker, loop)
    feed = TestDataFeed(TEST_DATA)
    try:
        await safe_broker.connect()
        live_strategy.on_start(wrapped_broker)
        await feed.start()
        async for timestamp, data, context in feed:
            for symbol, bar in data.items():
                broker.update_price(symbol, bar["close"])
                safe_broker.record_market_snapshot(symbol, bar["close"], timestamp)
            await asyncio.to_thread(live_strategy.on_data, timestamp, data, context, wrapped_broker)
        live_strategy.on_end(wrapped_broker)
    finally:
        feed.stop()
        await safe_broker.disconnect()
        state_path.unlink(missing_ok=True)

In [22]:
live_executed = False
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        run_async(run_live())
    live_executed = True
except RuntimeError as _e:
    if "Timeout" in str(_e) or "task" in str(_e).lower():
        print(f"Async live test skipped (Papermill environment): {_e}")
    else:
        raise

print("\nLive Results:")
print(f"   Features computed: {len(live_strategy.feature_log)}")
print(f"   Predictions made: {len(live_strategy.prediction_log)}")
print(f"   Signals generated: {len(live_strategy.signal_log)}")
print(f"   Orders submitted: {len(live_strategy.order_log)}")
if live_strategy.order_log:
    print(f"   Final cash: ${live_strategy.order_log[-1]['cash']:,.2f}")
    print(f"   Final positions: {live_strategy.order_log[-1]['positions']}")

INFO - SafeBroker initialized. Execution mode: shadow


INFO - SafeBroker reconciliation clean


INFO - VerifiableStrategy started


INFO - Shadow: Opened SPY LONG 19.0


INFO - SHADOW: buy 19.0 SPY @ $520.35 (value: $9,887)


...

INFO - SHADOW: sell 2.0 SPY @ $515.52 (value: $1,031)


INFO - Shadow: Opened QQQ LONG 25.0


INFO - SHADOW: buy 25.0 QQQ @ $387.86 (value: $9,697)


INFO - Strategy ended: 4 orders



Live Results:
   Features computed: 60
   Predictions made: 42
   Signals generated: 42
   Orders submitted: 4
   Final cash: $90,250.46
   Final positions: (('QQQ', 25),)


**发现：** 实盘风格重放在相同数据上生成第二条、完整插桩的流水线轨迹。这将回测与实盘的对比从直觉转化为可度量的比较。

**交易含义：** 技术一致性应通过相同输入和已记录的阶段输出进行验证，而不是从总体收益的粗略相似性中推断。


## 5. 验证测试

验证测试在每个阶段比较回测和实时输出。它们是决定一致性是保持还是被破坏的发布门禁。

In [23]:
TEST_NAMES = (
    "Feature Count Parity",
    "Feature Value Parity",
    "Prediction Parity",
    "Signal Parity",
    "Order Count Parity",
    "Order Fill Parity",
    "Cash Path Parity",
    "Position Path Parity",
    "Order ID Uniqueness",
)

被跳过的路径会产生明确的失败记录，保留原因而不晋级该候选。

In [24]:
def skipped_results() -> list[VerificationResult]:
    """Build the complete failed result set when either replay did not execute."""
    message = "SKIPPED: one replay did not execute, so parity cannot be evaluated"
    return [
        VerificationResult(
            test_name=name,
            passed=False,
            expected="N/A",
            actual="N/A",
            message=message,
            skipped=True,
            expected_difference=name in EXPECTED_DIFFERENCE,
        )
        for name in TEST_NAMES
    ]

计数一致性可防止基于 zip 的值比较掩盖缺失记录。

In [25]:
def count_result(name: str, reference: list[dict], candidate: list[dict]) -> VerificationResult:
    """Compare complete record counts for one pipeline stage."""
    return VerificationResult(
        test_name=name,
        passed=len(reference) == len(candidate),
        expected=len(reference),
        actual=len(candidate),
        message=f"Backtest: {len(reference)}, Live: {len(candidate)}",
    )

记录一致性会检查标识和值，并且仅对指定的浮点字段使用容差。

In [26]:
def matching_records(
    reference: list[dict],
    candidate: list[dict],
    fields: tuple[str, ...],
    float_fields: tuple[str, ...] = (),
) -> int:
    """Count ordered records whose selected fields match."""
    matches = 0
    for left, right in zip(reference, candidate, strict=False):
        same = True
        for field in fields:
            if field in float_fields:
                same &= abs(float(left[field]) - float(right[field])) < 1e-10
            else:
                same &= left[field] == right[field]
        matches += int(same)
    return matches

序列结果会使空日志或不相等日志失败，因此空洞的比较永远无法通过。

In [27]:
def sequence_result(
    name: str,
    reference: list[dict],
    candidate: list[dict],
    fields: tuple[str, ...],
    float_fields: tuple[str, ...] = (),
) -> VerificationResult:
    """Compare two complete, ordered stage logs."""
    matches = matching_records(reference, candidate, fields, float_fields)
    complete = bool(reference) and len(reference) == len(candidate)
    ratio = matches / len(reference) if complete else 0.0
    return VerificationResult(
        test_name=name,
        passed=ratio == 1.0,
        expected=1.0,
        actual=ratio,
        message=f"{matches}/{len(reference)} reference records match",
    )

每条路径还必须生成唯一的订单标识符，以便成交保持可审计。

In [28]:
def unique_order_ids_result(reference: list[dict], candidate: list[dict]) -> VerificationResult:
    """Require nonempty, unique order IDs independently on both execution paths."""
    reference_ids = [record["order_id"] for record in reference]
    candidate_ids = [record["order_id"] for record in candidate]
    passed = bool(reference_ids) and len(reference_ids) == len(set(reference_ids))
    passed &= bool(candidate_ids) and len(candidate_ids) == len(set(candidate_ids))
    return VerificationResult(
        test_name="Order ID Uniqueness",
        passed=passed,
        expected="unique IDs on both paths",
        actual=f"backtest={reference_ids}; live={candidate_ids}",
        message=f"Backtest: {len(set(reference_ids))}/{len(reference_ids)} unique; "
        f"Live: {len(set(candidate_ids))}/{len(candidate_ids)} unique",
    )

第一组结果涵盖提交前特征、预测和信号。

In [29]:
def pipeline_results(bt: VerifiableStrategy, live: VerifiableStrategy) -> list[VerificationResult]:
    """Compare the deterministic decision pipeline before order submission."""
    return [
        count_result("Feature Count Parity", bt.feature_log, live.feature_log),
        sequence_result(
            "Feature Value Parity",
            bt.feature_log,
            live.feature_log,
            ("timestamp", "symbol", "features"),
        ),
        sequence_result(
            "Prediction Parity",
            bt.prediction_log,
            live.prediction_log,
            ("timestamp", "symbol", "prediction"),
            ("prediction",),
        ),
        sequence_result(
            "Signal Parity",
            bt.signal_log,
            live.signal_log,
            ("timestamp", "symbol", "signal"),
        ),
    ]

第二组检查已提交订单、已实现成交和经纪商状态路径。

In [30]:
def execution_results(bt: VerifiableStrategy, live: VerifiableStrategy) -> list[VerificationResult]:
    """Compare order fills and post-fill broker state."""
    return [
        count_result("Order Count Parity", bt.order_log, live.order_log),
        sequence_result(
            "Order Fill Parity",
            bt.order_log,
            live.order_log,
            (
                "timestamp",
                "symbol",
                "signal",
                "size",
                "price",
                "status",
                "filled_quantity",
                "filled_price",
            ),
            ("price", "filled_price"),
        ),
        sequence_result("Cash Path Parity", bt.order_log, live.order_log, ("cash",), ("cash",)),
        sequence_result("Position Path Parity", bt.order_log, live.order_log, ("positions",)),
        unique_order_ids_result(bt.order_log, live.order_log),
    ]

发布门禁组合两组结果，并在任一重放被跳过时失败关闭。

In [31]:
def run_verification_tests() -> list[VerificationResult]:
    """Run the complete parity contract."""
    if not (backtest_executed and live_executed):
        return skipped_results()
    return pipeline_results(backtest_strategy, live_strategy) + execution_results(
        backtest_strategy, live_strategy
    )

人类可读报告会展示每一项检查，包括被跳过的阶段和有意设置为不同的阶段。

In [32]:
print("\n" + "=" * 60)
print("VERIFICATION RESULTS")
print("=" * 60)
results = run_verification_tests()


VERIFICATION RESULTS


紧凑的状态标签让详细表格在 notebook 和 CI 输出中保持可读。

In [33]:
def result_status(result: VerificationResult) -> str:
    if result.skipped:
        return "[SKIP]"
    if result.passed:
        return "[OK] PASS"
    if result.expected_difference:
        return "[EXPECTED] DIFF"
    return "[FAIL] FAIL"

In [34]:
for result in results:
    print(f"\n{result_status(result)}: {result.test_name}")
    print(f"   {result.message}")
    if not result.passed and not result.skipped:
        print(f"   Expected: {result.expected}")
        print(f"   Actual: {result.actual}")


[OK] PASS: Feature Count Parity
   Backtest: 60, Live: 60

[OK] PASS: Feature Value Parity
   60/60 reference records match

[OK] PASS: Prediction Parity
   42/42 reference records match

[OK] PASS: Signal Parity
   42/42 reference records match

[OK] PASS: Order Count Parity
   Backtest: 4, Live: 4

[OK] PASS: Order Fill Parity
   4/4 reference records match

[OK] PASS: Cash Path Parity
   4/4 reference records match

[OK] PASS: Position Path Parity
   4/4 reference records match

[OK] PASS: Order ID Uniqueness
   Backtest: 4/4 unique; Live: 4/4 unique


**发现：** 该验证代码块将多个一致性问题压缩为显式的通过/失败测试。
这将部署就绪状态转化为 CI 系统可以强制执行的产物。

**交易含义：** 如果一致性检查没有自动化，团队最终会在时间压力下跳过它们，
回测与实盘之间的差距就会以运营意外的形式再次出现。


## 6. 回归测试输出

该摘要按 CI 集成要求格式化，使 notebook 能充当可复现的部署门禁，
而不仅仅是叙述式讲解。

In [35]:
print("\n" + "=" * 60)
print("CI REGRESSION TEST SUMMARY")
print("=" * 60)

# The CI gate counts only tests that ran AND were not in EXPECTED_DIFFERENCE.
# Skipped tests and explicitly contracted differences do not consume a
# pass/fail slot. This harness currently expects no differences.
gate_results = [r for r in results if not r.skipped and not r.expected_difference]
passed = sum(1 for r in gate_results if r.passed)
failed = sum(1 for r in gate_results if not r.passed)
skipped = sum(1 for r in results if r.skipped)
expected_diff = sum(1 for r in results if r.expected_difference)

print(f"\nGate tests passed:  {passed}/{len(gate_results)}")
print(f"Gate tests failed:  {failed}/{len(gate_results)}")
print(f"Expected differences (informational): {expected_diff}")
print(f"Skipped:           {skipped}")

if skipped == len(results):
    print("\n[FAIL] LIVE PIPELINE NOT EXECUTED")
    print("   Parity could not be evaluated; this candidate cannot be promoted")
    exit_code = 1
elif failed == 0 and len(gate_results) > 0:
    print("\n[OK] ALL GATED VERIFICATION TESTS PASSED")
    print("   Technical parity confirmed between backtest and live pipelines")
    exit_code = 0
else:
    print("\n[FAIL] VERIFICATION FAILED")
    print("   Investigate failing tests before deploying live")
    for r in gate_results:
        if not r.passed:
            print(f"   - {r.test_name}: {r.message}")
    exit_code = 1

print(f"\nExit code: {exit_code}")


CI REGRESSION TEST SUMMARY

Gate tests passed:  9/9
Gate tests failed:  0/9
Expected differences (informational): 0
Skipped:           0

[OK] ALL GATED VERIFICATION TESTS PASSED
   Technical parity confirmed between backtest and live pipelines

Exit code: 0


**发现：** 这种 CI 风格的摘要将 notebook 结果转化为机器可读的发布门禁。

**交易含义：** 生产部署应只晋级那些能够向自动化工具无歧义地表达一致性
成功或失败的代码路径。


## Pytest 断言模式

库测试套件会直接断言相同的一致性条件。下面的
模式对应 `tests/live/test_parity.py` 在将本 notebook 晋级为 CI 测试框架的项目中
会包含的内容。

In [36]:
def test_parity_gate(results: list[VerificationResult]) -> None:
    """Pytest assertion pattern for the parity gate.

    A skipped pipeline blocks promotion because parity was not evaluated.
    EXPECTED_DIFFERENCE results are informational. Only gated results must pass.
    """
    if all(r.skipped for r in results):
        raise AssertionError("Live pipeline did not execute; parity gate is incomplete")

    gated = [r for r in results if not r.skipped and not r.expected_difference]
    failed = [r for r in gated if not r.passed]
    assert not failed, "Parity gate failed: " + ", ".join(r.test_name for r in failed)


if __name__ == "__main__":
    test_parity_gate(results)

In [37]:
print("\n" + "=" * 60)
print("PIPELINE VERIFICATION COMPLETE")
print("=" * 60)
if skipped == len(results):
    print("Result: [FAIL] live pipeline did not execute")
elif failed == 0:
    print(
        f"Result: [OK] {passed}/{len(gate_results)} gated tests passed; "
        f"{expected_diff} expected differences"
    )
else:
    print(f"Result: [FAIL] {failed}/{len(gate_results)} gated tests failed")


PIPELINE VERIFICATION COMPLETE
Result: [OK] 9/9 gated tests passed; 0 expected differences


## 关键要点

- **四个一致性阶段捕获不同类别的缺陷。** 特征一致性标识
  数据排序和浮点排序问题；预测一致性捕获模型
  版本管理或预处理漂移；信号一致性捕获阈值或
  持仓状态错误；订单一致性捕获头寸规模与舍入差异。
- **只要没有单独契约证明并非如此，不匹配就是失败。**
  两条路径消费同一行情带，因此特征计数、身份和值
  必须完全匹配；测试框架不会豁免预热差异。
- **跳过不等于通过。** 当实盘流水线无法执行时（Papermill
  异步约束），测试框架会报告 `FAIL`，而不是针对空的实盘日志通过一致性
  测试。候选版本会保持阻塞，直到两条路径都执行。
- **确定性仍必须覆盖决策路径。** 静态的逐标的
  种子偏移和固定的双 regime 行情带可在跨进程间复现，同时
  强制进行非空泛的 BUY 和 SELL 比较。

**下一步：** 将同一一致性测试框架扩展到真实特征流水线和保存的
模型产物；每当券商封装、头寸规模逻辑或
预处理代码发生变化时重新运行。